In [19]:
import geopandas
import pandas as pd

In [6]:
%cd /g/data/w42/dr6273/work/wind_drought

/g/data/w42/dr6273/work/wind_drought


In [7]:
!pwd

/g/data/w42/dr6273/work/wind_drought


In [57]:
# load 2026 resource estimates
resource = pd.read_csv('data/REZ_potential_2026.csv', index_col=0)

In [58]:
cols = [
    "potential_solar", "potential_wind",
    "existing_solar",
    "SG30_solar", "SG40_solar", "SG50_solar", 
    "SC30_solar", "SC40_solar", "SC50_solar", 
    "AT30_solar", "AT40_solar", "AT50_solar", 
    "existing_wind",
    "SG30_wind", "SG40_wind", "SG50_wind", 
    "SC30_wind", "SC40_wind", "SC50_wind", 
    "AT30_wind", "AT40_wind", "AT50_wind",
]

In [59]:
resource.columns = cols

In [60]:
# Split capacity in V3-4 across each
resource.loc["V3"] = resource.loc["V3-V4"] / 2
resource.loc["V4"] = resource.loc["V3-V4"] / 2
resource = resource.drop("V3-V4").sort_index()

In [63]:
# sort index by letter and number
resource = resource.loc[
    sorted(resource.index, key=lambda x: (x.rstrip("0123456789"), int(x.lstrip("NQSTV"))))
]

In [64]:
resource

,potential_solar,potential_wind,existing_solar,SG30_solar,SG40_solar,SG50_solar,SC30_solar,SC40_solar,SC50_solar,AT30_solar,...,existing_wind,SG30_wind,SG40_wind,SG50_wind,SC30_wind,SC40_wind,SC50_wind,AT30_wind,AT40_wind,AT50_wind
REZ,,,,,,,,,,,,,,,,,,,,,
N1,6385.0,0.0,166.0,100.0,100.0,150.0,250.0,250.0,250.0,150.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,150.0,150.0,150.0
N2,2985.0,7400.0,1256.0,0.0,0.0,950.0,200.0,2250.0,2250.0,200.0,...,652.0,50.0,50.0,3450.0,50.0,1800.0,6950.0,50.0,6300.0,7400.0
N3,4450.0,6850.0,2424.0,1800.0,1800.0,5500.0,1800.0,6700.0,6850.0,1800.0,...,2286.0,1350.0,1350.0,2300.0,1350.0,1750.0,2850.0,1350.0,2300.0,8600.0
N4,8000.0,5100.0,53.0,0.0,0.0,0.0,300.0,300.0,400.0,200.0,...,198.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,100.0,250.0
N5,5677.0,14000.0,1014.0,200.0,200.0,200.0,200.0,200.0,500.0,200.0,...,595.0,3250.0,3250.0,3250.0,3300.0,3300.0,3300.0,3250.0,3250.0,3250.0
N6,2146.0,1000.0,446.0,500.0,500.0,500.0,2150.0,2150.0,2150.0,1550.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
N7,100.0,400.0,0.0,0.0,0.0,0.0,100.0,100.0,100.0,0.0,...,270.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0
N8,350.0,300.0,0.0,0.0,0.0,0.0,350.0,350.0,350.0,0.0,...,113.0,0.0,0.0,0.0,0.0,0.0,200.0,100.0,300.0,500.0
N9,953.0,1400.0,0.0,1750.0,1750.0,1750.0,2050.0,2250.0,3700.0,1550.0,...,0.0,50.0,50.0,1300.0,300.0,1400.0,1500.0,1300.0,3950.0,6150.0


In [71]:
rez_gpd = geopandas.read_file(
    "data/indicative-rez-boundaries-2026-gis-data.kmz",
    driver="KMZ"
)

In [72]:
rez_gpd = rez_gpd[["Name", "description", "geometry"]]

In [73]:
rez_gpd.index = [i.split(" ")[0] for i in rez_gpd["Name"]]

In [74]:
rez_gpd.head()

,Name,description,geometry
Q1,Q1 Far North QLD,REZ,"POLYGON Z ((145.50712 -17.7214 3000, 145.65131..."
Q2,Q2 North Qld Clean Energy Hub,REZ,"POLYGON Z ((143.72015 -16.97938 0, 143.53034 -..."
Q3,Q3 Northern Qld,REZ,"POLYGON Z ((146.27782 -18.7192 0, 146.25228 -1..."
Q4,Q4 Isaac,REZ,"POLYGON Z ((148.49792 -22.15434 0, 148.36201 -..."
Q5,Q5 Barcaldine,REZ,"POLYGON Z ((145.82086 -23.84279 0, 145.9414 -2..."


In [75]:
rez_gpd = rez_gpd.join(resource, how="left")

In [76]:
rez_gpd.iloc[30]

Name                                           V8 Gippsland Offshore
description                                             Offshore REZ
geometry           GEOMETRYCOLLECTION Z (MULTIPOLYGON Z (((145.87...
potential_solar                                                  0.0
potential_wind                                               60000.0
existing_solar                                                   0.0
SG30_solar                                                       0.0
SG40_solar                                                       0.0
SG50_solar                                                       0.0
SC30_solar                                                       0.0
SC40_solar                                                       0.0
SC50_solar                                                       0.0
AT30_solar                                                       0.0
AT40_solar                                                       0.0
AT50_solar                        

In [77]:
rez_gpd.to_file(
    "data/resource_with_indicative_REZ_boundaries_2026.gpkg",
    layer="rez",
    driver="GPKG",
    engine="pyogrio",
    promote_to_multi=True,
)